# DR-KPT kernel Benchmarks

In [ ]:
!git clone https://github.com/houssamzenati/counterfactual-policy-mean-embedding code/

import sys
sys.path.insert(0, "code/counterfactual-policy-mean-embedding/src/testing")

In [ ]:
import numpy as np
from sklearn.metrics import pairwise_distances

from kpt import kernel_two_sample_test_reweight

from causal_medmnist import Scenario
from causal_medmnist.datasets import REGISTRY
from causal_medmnist.utils import sigmoid

In [ ]:
n_samples = 1000
num_repetitions = 100
n_permutations = 200
policy_clip = 0.05


def soft_policy(coefficients, shift):
    def policy(X):
        return np.clip(sigmoid(X @ coefficients + shift), policy_clip, 1 - policy_clip)
    return policy


def generate(scenario, n, rng, split):
    sample = scenario.generate(n, seed=int(rng.integers(0, 2**32)), split=split, replace=True)
    return sample.X, sample.A, sample.Y.reshape(n, -1), sample.propensity


def kpt_weights(A, pi_logging, p_pi, p_pi_prime):
    prob_logging = np.where(A == 1, pi_logging, 1 - pi_logging)
    w_pi = np.where(A == 1, p_pi, 1 - p_pi) / prob_logging
    w_pi_prime = np.where(A == 1, p_pi_prime, 1 - p_pi_prime) / prob_logging
    return w_pi, w_pi_prime


def run_test(dataset, n_samples, scale, shift, distributional_effect, rng):
    scenario = Scenario(dataset, effect_strength=scale, distributional_effect=distributional_effect)
    alpha = scenario.config.treatment_coefficients * scenario.confounding_strength
    pi = soft_policy(alpha, 0.0)
    pi_prime = soft_policy(alpha, shift)

    rejections = []
    for _ in range(num_repetitions):
        X, A, Y, pi_logging = generate(scenario, n_samples, rng, "train")
        w_pi, w_pi_prime = kpt_weights(A, pi_logging, pi(X), pi_prime(X))
        gamma = 1.0 / (np.median(pairwise_distances(Y)) ** 2)
        _, _, pval = kernel_two_sample_test_reweight(Y, w_pi, w_pi_prime, kernel_function="rbf", gamma=gamma, iterations=n_permutations, random_state=0)
        rejections.append(float(pval < 0.05))

    return np.mean(rejections)


def run(scale, shift, distributional_effect, rng):
    for dataset in sorted(REGISTRY):
        result = run_test(dataset, n_samples, scale, shift, distributional_effect, rng)
        print(f"[{dataset}][N={n_samples}][scale={scale}][shift={shift}] {result}")

In [ ]:
rng = np.random.default_rng(0)

run(scale=0.6, shift=2.0, distributional_effect=True, rng=rng)
run(scale=0.6, shift=0.0, distributional_effect=True, rng=rng)